## 1. Lab 13 project structure

The dashboard implementation uses these files:

```text
app.py
src/dashboard/__init__.py
src/dashboard/data_access.py
src/dashboard/layout.py
src/dashboard/callbacks.py
scripts/seed_mongo.py
Dockerfile
docker-compose.yml
.dockerignore
requirements-dashboard.txt
```

In [3]:
import os
import sys
from pathlib import Path

current = Path.cwd().resolve()

for path in [current] + list(current.parents):
    if (path / "src").exists() and (path / "app.py").exists():
        PROJECT_ROOT = path
        break
else:
    raise RuntimeError("Project root not found. Open the notebook from the project folder.")

os.chdir(PROJECT_ROOT)

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Working directory:", Path.cwd())

Project root: C:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD
Working directory: C:\Users\38760\Desktop\Unstructured_Data\News_Media_Monitoring_Pipeline_UD


In [4]:
from pathlib import Path
import os
import sys
import subprocess
import pandas as pd

PROJECT_ROOT = Path.cwd()

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

required_paths = [
    "app.py",
    "src/dashboard/__init__.py",
    "src/dashboard/data_access.py",
    "src/dashboard/layout.py",
    "src/dashboard/callbacks.py",
    "scripts/seed_mongo.py",
    "Dockerfile",
    "docker-compose.yml",
    ".dockerignore",
    "requirements-dashboard.txt",
    "data/processed/cleaned/cleaned_data.csv",
]

structure_df = pd.DataFrame(
    [{"path": path, "exists": Path(path).exists()} for path in required_paths]
)

display(structure_df)

,path,exists
0,app.py,True
1,src/dashboard/__init__.py,True
2,src/dashboard/data_access.py,True
3,src/dashboard/layout.py,True
4,src/dashboard/callbacks.py,True
5,scripts/seed_mongo.py,True
6,Dockerfile,True
7,docker-compose.yml,True
8,.dockerignore,True
9,requirements-dashboard.txt,True


## 2. Load dashboard data

The dashboard data access layer loads from MongoDB when available and falls back to the cleaned CSV. This verifies that the dataset is available to the dashboard.

In [5]:
from src.dashboard.data_access import (
    load_dashboard_data,
    get_available_categories,
    get_available_document_types,
    get_year_range,
    get_summary_metrics,
    filter_news_data,
)

df = load_dashboard_data()

print("Dataset shape:", df.shape)
print("Year range:", get_year_range(df))
print("Categories:", get_available_categories(df)[:10])
print("Document types:", get_available_document_types(df)[:10])
print("Summary metrics:", get_summary_metrics(df))

display(df[["title", "category", "document_type", "published_year", "rating_score", "popularity", "content_length"]].head(10))

Dataset shape: (1318, 35)
Year range: (2026, 2026)
Categories: ['business', 'culture', 'encoding_test', 'excel_summary', 'json', 'news_api', 'ocr_image', 'ocr_pdf', 'pdf', 'pdf_two_column']
Document types: ['encoding_test', 'excel', 'excel_summary', 'json', 'news_api', 'ocr_image', 'ocr_pdf', 'pdf', 'pdf_two_column', 'scraped_html']
Summary metrics: {'total_records': 1318, 'category_count': 19, 'document_type_count': 15, 'avg_rating': 0.82, 'avg_popularity': 92.59, 'avg_content_length': 92.55}


,title,category,document_type,published_year,rating_score,popularity,content_length
0,Election Update,politics,excel,2026,4.0,34.0,15
1,AI Market Growth,business,excel,2026,7.0,28.0,16
2,Sports Highlights,sports,excel,2026,5.0,19.0,17
3,Climate Policy Shift,politics,excel,2026,3.0,23.0,20
4,Café Culture Trends,culture,excel,2026,6.0,14.0,19
5,München Startup Round,technology,excel,2026,8.0,17.0,21
6,Žurnal Election Pulse,politics,excel,2026,5.0,21.0,21
7,Streaming Rights Watch,business,excel,2026,4.0,26.0,22
8,Weekend Match Buzz,sports,excel,2026,7.0,31.0,18
9,Inflation Briefing,business,excel,2026,2.0,22.0,18


## 3. Test dashboard filtering logic

This verifies that the same filtering logic used by the Dash callbacks works for category, document type, year range, and search text.

In [6]:
filtered_df = filter_news_data(
    df,
    categories=["business"],
    document_types=[],
    year_range=get_year_range(df),
    search_text="AI",
)

print("Filtered shape:", filtered_df.shape)
display(filtered_df[["title", "category", "document_type", "rating_score", "popularity"]].head(10))

Filtered shape: (1, 35)


,title,category,document_type,rating_score,popularity
0,AI Market Growth,business,excel,7.0,28.0


## 4. Verify Dash app import

The root `app.py` exposes both the Dash application object and the Flask `server` object required by Gunicorn.

In [7]:
from app import app, server

print("App title:", app.title)
print("Server exists:", server is not None)
print("Layout type:", type(app.layout))
print("Registered callbacks:", len(app.callback_map))

App title: News Media Monitoring Dashboard
Server exists: True
Layout type: <class 'dash.html.Div.Div'>
Registered callbacks: 8


## 5. Verify dashboard modules

This confirms that the modular Lab 13 dashboard files import correctly.

In [8]:
from src.dashboard.layout import create_layout
from src.dashboard.callbacks import register_callbacks
from src.dashboard.data_access import get_collection_config

layout = create_layout()
config = get_collection_config()

print("Layout type:", type(layout))
print("register_callbacks callable:", callable(register_callbacks))
print("MongoDB config:", config)

Layout type: <class 'dash.html.Div.Div'>
register_callbacks callable: True
MongoDB config: {'mongo_uri': 'mongodb://localhost:27017', 'db_name': 'news_dashboard', 'collection_name': 'news_records', 'csv_path': 'data\\processed\\cleaned\\cleaned_data.csv'}


## 6. Seed MongoDB

Run this when MongoDB is running locally or through Docker Compose. The script loads the cleaned dataset into the `news_dashboard.news_records` collection and creates useful indexes.

In [9]:
def run_command(command, timeout=180):
    try:
        result = subprocess.run(
            command,
            shell=True,
            text=True,
            capture_output=True,
            timeout=timeout,
        )
        return {
            "command": command,
            "return_code": result.returncode,
            "stdout": result.stdout.strip(),
            "stderr": result.stderr.strip(),
        }
    except Exception as exc:
        return {
            "command": command,
            "return_code": -1,
            "stdout": "",
            "stderr": repr(exc),
        }

seed_result = run_command("python scripts/seed_mongo.py", timeout=180)
seed_df = pd.DataFrame([seed_result])
display(seed_df)

,command,return_code,stdout,stderr
0,python scripts/seed_mongo.py,0,MongoDB seeding complete\nCSV path: data\proce...,


## 7. Local dashboard run command

Run this in a terminal, not inside the notebook, because the Dash server is a long-running process:

```powershell
python app.py
```

Then open:

```text
http://127.0.0.1:8050
```

Expected result: the News Media Monitoring Dashboard loads with KPI cards, filters, charts, a table, and the live ticker.

## 8. Docker build and run commands

Run these commands in PowerShell from the project root:

```powershell
docker compose down
docker compose build --no-cache web
docker compose up -d
docker compose ps
python scripts/seed_mongo.py
```

Then open:

```text
http://localhost:8050
```

## 9. Docker verification

This cell runs lightweight Docker verification commands. If Docker Desktop is not running, it prints the error and continues.

In [10]:
commands = [
    'docker compose ps',
    'docker exec news-dashboard-mongo mongosh news_dashboard --quiet --eval "db.news_records.countDocuments()"',
    'docker exec news-dashboard-app python -c "from src.dashboard.data_access import load_dashboard_data; df=load_dashboard_data(); print(df.shape)"',
]

verification_results = [run_command(command) for command in commands]
verification_df = pd.DataFrame(verification_results)
display(verification_df)

,command,return_code,stdout,stderr
0,docker compose ps,0,NAME IMAGE COMMAND SERVICE CREATE...,
1,docker exec news-dashboard-mongo mongosh news_...,1,,Error response from daemon: No such container:...
2,"docker exec news-dashboard-app python -c ""from...",1,,Error response from daemon: No such container:...


## 10. Pipeline integration check

The full pipeline should not automatically start the Dash web server because the dashboard is a long-running app. Instead, the Lab 13 pipeline stage prints the commands needed to launch and verify the dashboard.

In [11]:
from src.pipeline.run_pipeline import run_lab13_dashboard_stage

stage_result = run_lab13_dashboard_stage()
stage_result


Lab 13 dashboard is ready.

Local run:
  python scripts/seed_mongo.py
  python app.py
  Open http://127.0.0.1:8050

Docker run:
  docker compose up -d
  python scripts/seed_mongo.py
  Open http://localhost:8050

Docker verification:
  docker compose ps
  docker compose logs web --tail 50
  docker exec news-dashboard-mongo mongosh news_dashboard --quiet --eval "db.news_records.countDocuments()"
  docker exec news-dashboard-app python -c "from src.dashboard.data_access import load_dashboard_data; df=load_dashboard_data(); print(df.shape)"

Stop Docker:
  docker compose down



{'app_entrypoint': 'app.py',
 'local_url': 'http://127.0.0.1:8050',
 'docker_url': 'http://localhost:8050',
 'seed_command': 'python scripts/seed_mongo.py',
 'docker_start_command': 'docker compose up -d',
 'docker_stop_command': 'docker compose down',
 'mongo_database': 'news_dashboard',
 'mongo_collection': 'news_records'}

## 12. Lab 13 completion summary

The News Media Monitoring Pipeline dashboard includes:

- Dash application entry point in `app.py`
- Modular dashboard package in `src/dashboard/`
- MongoDB data access with CSV fallback
- KPI cards
- Category, document type, year, and search filters
- Interactive Plotly charts
- Top matching records table
- Simulated live ticker using `dcc.Interval`
- MongoDB seed script
- Dockerfile using Gunicorn
- Docker Compose with Dash app and MongoDB services
- Dashboard-specific Docker requirements file
- Pipeline integration stage

Final expected Docker checks:

```text
news-dashboard-app is running on port 8050
news-dashboard-mongo is healthy
MongoDB count is 1318
Dashboard data shape is (1318, 35)
```